# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [3]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

for pkg in ["stopwords", "punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean(text):
    text = str(text).strip().lower()
    text = "".join(ch for ch in text if not ch.isdigit())
    for p in string.punctuation:
        text = text.replace(p, " ")
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

data["clean_text"] = data["text"].apply(clean)
data[["text", "clean_text"]].head()

,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gld cunixb columbia edu gary dare subject stan...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,miner kuhub ukans edu subject ancient book org...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivov superior carleton vladimir zhivov subj...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = TfidfVectorizer(max_df=0.75, min_df=5)
data_vectorized = vectorizer.fit_transform(data["clean_text"])
print("Shape:", data_vectorized.shape)

lda_model = LatentDirichletAllocation(n_components=2, max_iter=100, random_state=42)
lda_model.fit(data_vectorized)

Shape: (1199, 4800)


LatentDirichletAllocation(max_iter=100, n_components=2, random_state=42)

##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [6]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [7]:
print_topics(lda_model, vectorizer)

Topic 0:
[('game', 28.800108130785237), ('team', 27.789407215367802), ('edu', 27.282222453129705), ('hockey', 20.92097912954489), ('player', 19.535478989168187), ('play', 16.189966595684922), ('year', 15.029629685672965), ('nhl', 14.99654503294577), ('university', 14.093485737098248), ('playoff', 14.050063776582318)]
Topic 1:
[('god', 39.83964023520142), ('christian', 26.286805419368292), ('edu', 24.519454232760477), ('jesus', 21.1605169553011), ('people', 19.363831849993165), ('would', 18.538216548895086), ('church', 18.53556217069679), ('one', 18.313667355977916), ('bible', 15.216916655509188), ('believe', 14.837082499783877)]


In [8]:
lda_3 = LatentDirichletAllocation(n_components=3, max_iter=100, random_state=42).fit(data_vectorized)
print_topics(lda_3, vectorizer)

Topic 0:
[('game', 28.64186729552366), ('team', 27.631084481851858), ('edu', 27.132130821927458), ('hockey', 20.763652684936563), ('player', 19.377146297383938), ('play', 16.00516470570368), ('nhl', 14.836313566203566), ('year', 14.605651199463118), ('playoff', 13.890238089037751), ('university', 13.830146533425214)]
Topic 1:
[('god', 39.67960584459507), ('christian', 26.12269405733602), ('edu', 24.333687612540253), ('jesus', 21.003393298924852), ('church', 18.375067936104458), ('people', 18.361235448914353), ('would', 17.972341053060912), ('one', 17.696450926198338), ('bible', 15.05479089700107), ('believe', 14.213551192500347)]
Topic 2:
[('familiar', 0.3382361173836385), ('pub', 0.33819726400348454), ('testing', 0.3380769569495339), ('ann', 0.33785993210468185), ('verse', 0.33766902307391794), ('distribution', 0.3376686692098663), ('quinn', 0.33757600709015667), ('sig', 0.3375356943995059), ('skidmore', 0.3375338577577597), ('max', 0.33753290114226087)]


## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [9]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [10]:
example_vectorized = vectorizer.transform([clean(t) for t in example])

topic_mix = lda_model.transform(example_vectorized)
print("Konu karışımı:", topic_mix.round(3))
print("En olası konu: Topic", topic_mix.argmax())

Konu karışımı: [[0.853 0.147]]
En olası konu: Topic 0


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!